# 28 — Case-surge definition review and warning sensitivity

Primary definition unchanged. Reuses current three-model2020predictions, no refit, no2021–2023data. This is retrospective diagnostic analysis. Five one-factor scenarios: primary(5cases,2SD), min3, min10,1.5SD,2.5SD. Each scenario calibrates warnings in Apr–Aug before Sep–Dec assessment.

Read saved outputs first using the last cell. Run the analysis cell only if rebuilding. Different event labels change the task, prevalence and event denominator; do not select labels for the best AP.

In [ ]:
from pathlib import Path
ROOT=Path.cwd()
if (ROOT/"Weikun").is_dir(): ROOT=ROOT/"Weikun"
OUTPUT=ROOT/"outbreak_review_outputs"/"case_surge_review_20261101"
print(OUTPUT.resolve())

In [ ]:
from pathlib import Path
import json,hashlib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import average_precision_score,precision_recall_curve

def event_labels(counts, min_cases=5, sd_multiplier=2):
    """Causal labels; onset is confirmation, never backdated."""
    s=pd.Series(np.asarray(counts,dtype=float))
    ma=s.rolling(3,min_periods=3).mean()
    hist=ma.shift(3).rolling(28,min_periods=28)
    mean,sd=hist.mean(),hist.std(ddof=1)
    threshold=mean+sd_multiplier*sd
    qualify=(ma>threshold)&(s.rolling(3).sum()>=min_cases)&threshold.notna()
    valid=threshold.notna().to_numpy()
    # Smooth continuous excess with the same sign as the two qualification gates.
    scale=np.maximum(sd.to_numpy(),1.)
    margin=np.minimum((ma.to_numpy()-threshold.to_numpy())/scale,
                      (s.rolling(3).sum().to_numpy()-min_cases+1e-9)/(3*scale))
    active=np.zeros(len(s),bool);onset=np.zeros(len(s),bool)
    in_event=False;off=0
    for k in range(len(s)):
        if in_event:
            off=0 if qualify.iloc[k] else off+1
            if off>=7:in_event=False;off=0
        elif k>0 and qualify.iloc[k] and qualify.iloc[k-1]:
            in_event=True;onset[k]=True
        active[k]=in_event
    return dict(ma=ma.to_numpy(),threshold=threshold.to_numpy(),sd=sd.to_numpy(),
                qualify=qualify.to_numpy(),valid=valid,margin=margin,active=active,onset=onset)

def warning_events(score,eligible,truth_onset,origins,dates,threshold):
    """Seven-day issue cooldown. One alarm matches at most one event."""
    rows=[];events=[]
    for z in range(len(score)):
        event_dates=dates[truth_onset[z]]
        # Denominator: events with at least one scored eligible origin 1..3 days before.
        covered=[d for d in event_dates if np.any(eligible[z]&((d-origins).astype('timedelta64[D]').astype(int)>=1)&((d-origins).astype('timedelta64[D]').astype(int)<=3))]
        matched=set();last=None
        for k,d in enumerate(origins):
            if not eligible[z,k] or not score[z,k]>threshold:continue
            if last is not None and (d-last).astype('timedelta64[D]').astype(int)<7:continue
            last=d
            candidates=[e for e in covered if e not in matched and 1<=int((e-d)/np.timedelta64(1,'D'))<=3]
            target=min(candidates) if candidates else None
            if target is not None:matched.add(target)
            rows.append(dict(region_index=z,forecast_origin=str(d)[:10],matched_event_date=str(target)[:10] if target is not None else '',
                             true_warning=target is not None,lead_days=float((target-d)/np.timedelta64(1,'D')) if target is not None else np.nan))
        for d in covered:events.append(dict(region_index=z,event_date=str(d)[:10],detected=d in matched))
    alarms=pd.DataFrame(rows,columns=['region_index','forecast_origin','matched_event_date','true_warning','lead_days'])
    ev=pd.DataFrame(events,columns=['region_index','event_date','detected'])
    tp=int(alarms.true_warning.sum()) if len(alarms) else 0;fp=len(alarms)-tp
    exposure=int(eligible.sum())
    metrics=dict(events=len(ev),detected_events=int(ev.detected.sum()) if len(ev) else 0,warning_episodes=len(alarms),false_warning_episodes=fp,
                 event_recall=tp/len(ev) if len(ev) else np.nan,warning_precision=tp/len(alarms) if len(alarms) else np.nan,
                 false_warnings_per_30_at_risk_zip_days=fp*30/exposure if exposure else np.nan,
                 mean_lead_days=float(alarms.lead_days.mean()) if len(alarms) else np.nan,
                 median_lead_days=float(alarms.lead_days.median()) if len(alarms) else np.nan,at_risk_zip_days=exposure)
    return metrics,alarms,ev

def run_outbreak(root,output,min_cases=5,sd_multiplier=2):
    root,output=Path(root),Path(output);output.mkdir(parents=True,exist_ok=True);fig=output/'figures';fig.mkdir(exist_ok=True)
    paths={'Dynamic Panel FE':root/'rolling_trajectory_outputs/rolling60_weekly_development_20261027/predictions.npz',
           'XGBoost':root/'xgboost_trajectory_outputs/xgb60_weekly_development_20261027/predictions.npz',
           'Random Forest':root/'random_forest_trajectory_outputs/rf60_weekly_development_20261027/predictions.npz'}
    arrays={m:np.load(p,allow_pickle=False) for m,p in paths.items()};ref=next(iter(arrays.values()))
    ids=ref['zip'];origins=ref['forecast_origins'].astype('datetime64[D]')
    for a in arrays.values():
        for k in ['zip','forecast_origins','horizons','actual_cases']:assert np.array_equal(a[k],ref[k]),k
    source=root/'rolling_trajectory_outputs/inputs/nj_daily_features.csv.gz'
    frame=pd.read_csv(source,usecols=['zip','date','cases_today'],dtype={'zip':'string'},parse_dates=['date'])
    frame=frame.loc[frame.zip.isin(ids)&frame.date.between('2020-01-01','2020-12-31')]
    assert not frame.duplicated(['zip','date']).any()
    pivot=frame.pivot(index='zip',columns='date',values='cases_today').reindex(ids)
    dates=pivot.columns.to_numpy().astype('datetime64[D]');counts=pivot.to_numpy()
    assert np.isfinite(counts).all() and (counts>=0).all()
    assert np.all(np.diff(dates)==np.timedelta64(1,'D'))
    pos=np.searchsorted(dates,origins)
    actual=np.stack([counts[:,pos+h] for h in [1,2,3]],axis=-1)
    assert np.array_equal(actual,ref['actual_cases'])
    labs=[event_labels(row,min_cases,sd_multiplier) for row in counts]
    onset=np.stack([x['onset'] for x in labs]);active=np.stack([x['active'] for x in labs]);valid=np.stack([x['valid'] for x in labs])
    eligible=(~active[:,pos])&valid[:,pos]
    future=np.stack([onset[:,pos+h] for h in [1,2,3]],axis=-1).any(axis=-1)
    daily=pd.DataFrame({'zip':np.repeat(ids,len(dates)),'date':np.tile(dates,len(ids)),'cases_today':counts.ravel()})
    for name in ['ma','threshold','qualify','active','onset','valid']:daily[name]=np.stack([x[name] for x in labs]).ravel()
    daily.to_csv(output/'daily_event_labels_2020.csv.gz',index=False,compression='gzip')
    eventrows=[]
    for z in range(len(ids)):
        for k in np.flatnonzero(onset[z]):eventrows.append(dict(zip=ids[z],event_date=str(dates[k]),cases_today=counts[z,k],mean3=labs[z]['ma'][k],threshold=labs[z]['threshold'][k]))
    events=pd.DataFrame(eventrows);events.to_csv(output/'event_catalog_2020.csv',index=False)
    byzip=pd.DataFrame({'zip':ids,'events':onset.sum(axis=1),'case_total_2020':counts.sum(axis=1)})
    byzip.to_csv(output/'event_counts_by_zip.csv',index=False)
    events.assign(quarter=pd.to_datetime(events.event_date).dt.to_period('Q').astype(str)).groupby('quarter').size().rename('events').to_csv(output/'event_counts_by_quarter.csv')
    # Verify causality: truncating at a date cannot change any earlier labels.
    for z in [0,len(ids)//2,len(ids)-1]:
        for cutoff in [120,240,330]:
            cut=event_labels(counts[z,:cutoff],min_cases,sd_multiplier)
            for key in ['qualify','active','onset']:assert np.array_equal(cut[key],labs[z][key][:cutoff])
    scores={}
    for model,a in arrays.items():
        pred=np.maximum(a['raw_predictions'],0)
        score=np.full(eligible.shape,np.nan)
        # Future thresholds use only pre-origin actuals and the issued predictions.
        for j,k in enumerate(pos):
            hist=counts[:,k-32:k+1]
            seq=np.concatenate([hist,pred[:,j,:]],axis=1)
            means=np.full(seq.shape,np.nan)
            means[:,2:]=(seq[:,2:]+seq[:,1:-1]+seq[:,:-2])/3
            margins=[]
            for d in range(seq.shape[1]-4,seq.shape[1]):
                baseline=means[:,d-30:d-2]
                assert baseline.shape[1]==28 and np.isfinite(baseline).all()
                mu=baseline.mean(axis=1);sd=baseline.std(axis=1,ddof=1);scale=np.maximum(sd,1.)
                margins.append(np.minimum((means[:,d]-mu-sd_multiplier*sd)/scale,(seq[:,d-2:d+1].sum(axis=1)-min_cases+1e-9)/(3*scale)))
            margins=np.stack(margins,axis=1)
            score[:,j]=np.maximum.reduce([np.minimum(margins[:,h-1],margins[:,h]) for h in [1,2,3]])
            if j==0:
                for z in [0,100,584]:
                    check=event_labels(seq[z],min_cases,sd_multiplier)['margin'][-4:]
                    assert np.allclose(margins[z],check,atol=1e-7), 'vector risk mismatch'
        assert np.isfinite(score).all()
        scores[model]=score
    calibration=(origins>=np.datetime64('2020-04-30'))&(origins<=np.datetime64('2020-08-28'))
    assert (origins[calibration].max()+np.timedelta64(3,'D'))<np.datetime64('2020-09-01')
    assessment=(origins>=np.datetime64('2020-09-01'))&(origins<=np.datetime64('2020-12-28'))
    metrics=[];thresholds=[];curves=[]
    for model,score in scores.items():
        # Fixed rule-based candidates; zero means the forecast satisfies both gates.
        choices=[]
        for q in [0,.25,.5,1,2,3,5,10,np.inf]:
            met,_,_=warning_events(score[:,calibration],eligible[:,calibration],onset,origins[calibration],dates,q)
            choices.append(dict(model=model,threshold=q,**met))
        allowed=[x for x in choices if x['false_warnings_per_30_at_risk_zip_days']<=1]
        chosen=min(allowed,key=lambda x:x['threshold']);thresholds.extend(choices)
        for phase,mask in [('calibration',calibration),('assessment',assessment)]:
            lab=future[:,mask][eligible[:,mask]];sc=score[:,mask][eligible[:,mask]]
            met,alarms,ev=warning_events(score[:,mask],eligible[:,mask],onset,origins[mask],dates,chosen['threshold'])
            ap=average_precision_score(lab,sc) if lab.any() else np.nan
            metrics.append(dict(model=model,phase=phase,threshold=chosen['threshold'],average_precision=ap,
                                positive_origin_share=float(lab.mean()),positive_origins=int(lab.sum()),**met))
            tag=model.lower().replace(' ','_')+'_'+phase
            alarms['zip']=alarms.region_index.map(dict(enumerate(ids)));ev['zip']=ev.region_index.map(dict(enumerate(ids)))
            alarms.to_csv(output/(tag+'_warning_episodes.csv'),index=False);ev.to_csv(output/(tag+'_event_detection.csv'),index=False)
            if phase=='assessment' and lab.any():
                precision,recall,_=precision_recall_curve(lab,sc);curves.append((model,precision,recall,ap,float(lab.mean())))
        pd.DataFrame({'zip':np.repeat(ids,len(origins)),'forecast_origin':np.tile(origins,len(ids)),
                      'eligible':eligible.ravel(),'event_in_next_3_days':future.ravel(),'risk_score':score.ravel()}).to_csv(output/(model.lower().replace(' ','_')+'_risk_scores.csv.gz'),index=False,compression='gzip')
    result=pd.DataFrame(metrics);result.to_csv(output/'warning_metrics.csv',index=False)
    pd.DataFrame(thresholds).to_csv(output/'calibration_threshold_grid.csv',index=False)
    plt.rcParams.update({'font.family':'Arial','font.size':11,'axes.spines.top':False,'axes.spines.right':False})
    f,ax=plt.subplots(1,2,figsize=(11,4));ax[0].hist(byzip.events,bins=np.arange(byzip.events.max()+2)-.5,color='#2563eb');ax[0].set(xlabel='Confirmed events per ZIP in 2020',ylabel='ZIPs')
    q=events.assign(q=pd.to_datetime(events.event_date).dt.to_period('Q').astype(str)).groupby('q').size();ax[1].bar(q.index,q,color='#087f78');ax[1].set(ylabel='Confirmed events',title='Observed rule-defined event counts');f.tight_layout();f.savefig(fig/'01_event_distribution.png',dpi=180);plt.close(f)
    f,ax=plt.subplots(figsize=(7,5))
    for m,p,r,ap,prior in curves:ax.plot(r,p,label=f'{m}: AP={ap:.3f}')
    if curves:ax.axhline(curves[0][-1],ls='--',color='gray',label=f'Positive share={curves[0][-1]:.3f}')
    ax.set(xlabel='Recall',ylabel='Precision',title='Sep-Dec 2020 exploratory assessment');ax.legend();f.tight_layout();f.savefig(fig/'02_precision_recall.png',dpi=180);plt.close(f)
    f,axes=plt.subplots(1,2,figsize=(11,4));a=result[result.phase.eq('assessment')]
    axes[0].bar(a.model,a.event_recall,color=['#64748b','#2563eb','#087f78']);axes[0].set(ylabel='Event recall',ylim=(0,1));axes[1].bar(a.model,a.false_warnings_per_30_at_risk_zip_days,color=['#64748b','#2563eb','#087f78']);axes[1].axhline(1,ls='--',color='gray');axes[1].set(ylabel='False warnings /30 at-risk ZIP-days');f.tight_layout();f.savefig(fig/'03_warning_tradeoff.png',dpi=180);plt.close(f)
    protocol=dict(name='Rule-defined NJ case-surge task v1; not official outbreak labels',baseline='Trailing3-day mean; preceding28 MA3 values excluding current and last2 MA3 values',threshold=f'baseline mean +{sd_multiplier} sample SD; minimum3-day sum{min_cases}; strict exceedance',confirmation='2 consecutive qualifying days; onset=second day, never backdate',reset='7 consecutive nonqualifying days; onset cannot recur while active',forecast='Existing raw direct1/2/3-day predictions clipped at0; no refit',risk='max over future confirmation days of min(two consecutive qualification margins); normalized by max(baselineSD,1); count gate incorporated',eligibility='Origin inactive and baseline defined; predicts new confirmation within1..3days',warning='score > calibrated threshold,7-day cooldown after each issued warning',calibration='Origins2020-04-30..2020-08-28; all targets <=Aug31; Aug29-31 origins omitted',assessment='2020-09-01..2020-12-28 exploratory; previously inspected',threshold_selection='smallest fixed nonnegative grid threshold with <=1 false warning per30 at-risk ZIP-days in calibration; no assessment tuning',AP='sklearn average_precision_score, step-integrated PR summary; not trapezoidal area',event_matching='earliest issued alert1..3days before confirmation; one-to-one; denominators only events with eligible scored origins',lead_time='reported only for detected events; misses and false alarms reported alongside',limitations=['Research proxy labels, not independently adjudicated outbreaks','Onset records missing excluded; retrospective availability assumption','Static2021 covariates and unvalidated ZIP-ZCTA linkage','2020 thresholds/definitions are exploratory; later assessment is not an untouched test'],sources={m:dict(path=str(p),sha256=hashlib.sha256(p.read_bytes()).hexdigest()) for m,p in paths.items()})
    protocol['min_cases']=min_cases;protocol['sd_multiplier']=sd_multiplier
    (output/'task_definition.json').write_text(json.dumps(protocol,indent=2),encoding='utf-8')
    summary=dict(regions=len(ids),origins=len(origins),events_2020=len(events),zips_with_events=int((byzip.events>0).sum()),qualifying_zip_days=int(sum(x['qualify'].sum() for x in labs)),matched_targets_verified=True,causal_prefix_checks_passed=True,no_model_refit=True,calibration_targets_end=str(origins[calibration].max()+np.timedelta64(3,'D')),assessment_origins_start=str(origins[assessment].min()))
    (output/'verification.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
    report='# 病例激增任务v1：探索性结果\n\n'+json.dumps(summary,ensure_ascii=False,indent=2)+'\n\n'+result.to_string(index=False)+'\n\n先看事件数量与分布，再看assessment行。AP的参考是positive_origin_share，不是0.5。Lead time只描述检出的事件，不能忽略漏报。报警阈值只在4—8月选择；9—12月不调阈值。事件起点为第二次满足条件的确认日，不是生物学真实爆发日。误报预算按非活动、可评分的ZIP-days归一化。\n\n如果最小阈值0已满足宽松的1/30误报预算，就保留0，不根据assessment寻找更好的阈值。无事件的地区也必须保留。后续先审查事件例子与敏感性，再决定冻结方案；不把研究定义当成官方疫情判定。'
    report+='\n\n注意：确认日需要连续两天，因此提前1天的成功预警可能已有首个异常日作为输入。它不是在任何异常出现前的预警；也不能解释为已验证感染爆发。事件基线采用28天移动均值，持续缓慢上升可能被适应性基线吸收，数值依赖该代理定义。校准origin截至8/28，使三天目标不跨9/1。\n'
    (output/'RESULTS_CN.md').write_text(report,encoding='utf-8')
    print(json.dumps(summary));print(result.to_string(index=False));return result


import contextlib,io
import matplotlib.dates as mdates

MODELS=['Dynamic Panel FE','XGBoost','Random Forest']
COLORS=['#64748b','#2563eb','#087f78']
def tag(model):return model.lower().replace(' ','_')
def run_review(root,output,primary=None):
    root,output=Path(root),Path(output);output.mkdir(parents=True,exist_ok=True)
    primary=Path(primary) if primary else root/'outbreak_task_outputs/case_surge_v1_20261101'
    figures=output/'figures';figures.mkdir(exist_ok=True)
    # Immutable primary inputs: preserve Notebook27 output and record its hashes.
    source_hash={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in [primary/'warning_metrics.csv',primary/'daily_event_labels_2020.csv.gz',primary/'task_definition.json']}
    configs=[('PRIMARY',5,2),('MIN3',3,2),('MIN10',10,2),('SD1_5',5,1.5),('SD2_5',5,2.5)]
    all_metrics=[];catalog=[]
    for name,minimum,sd in configs:
        print('Scenario',name,'minimum',minimum,'SD',sd,flush=True)
        folder=output/'scenarios'/name
        with contextlib.redirect_stdout(io.StringIO()):res=run_outbreak(root,folder,minimum,sd)
        res['scenario']=name;res['min_cases']=minimum;res['sd_multiplier']=sd;all_metrics.append(res)
        ev=pd.read_csv(folder/'event_catalog_2020.csv',dtype={'zip':'string'})
        catalog.append(dict(scenario=name,min_cases=minimum,sd_multiplier=sd,events_2020=len(ev),zips_with_events=ev.zip.nunique()))
        if name=='PRIMARY':
            old=pd.read_csv(primary/'warning_metrics.csv').sort_values(['model','phase']).reset_index(drop=True)
            new=res.sort_values(['model','phase']).reset_index(drop=True)
            for col in ['average_precision','event_recall','warning_precision','mean_lead_days','events','false_warning_episodes']:
                assert np.allclose(old[col],new[col],equal_nan=True),('Primary replication mismatch',col)
    metrics=pd.concat(all_metrics,ignore_index=True);metrics.to_csv(output/'sensitivity_metrics.csv',index=False)
    pd.DataFrame(catalog).to_csv(output/'sensitivity_event_counts.csv',index=False)
    assess=metrics.loc[metrics.phase.eq('assessment')].copy()
    for col in ['average_precision','event_recall','warning_precision']:
        assess[col+'_rank']=assess.groupby('scenario')[col].rank(ascending=False,method='min')
    assess.to_csv(output/'assessment_sensitivity_and_ranks.csv',index=False)
    # Primary lead-time review: report recall against ALL eligible events, not only detections.
    daily=pd.read_csv(primary/'daily_event_labels_2020.csv.gz',dtype={'zip':'string'},parse_dates=['date'])
    daily=daily.sort_values(['zip','date']).set_index(['zip','date'])
    leadrows=[];eventdetails=[];warningdetails=[]
    for model in MODELS:
        alarms=pd.read_csv(primary/(tag(model)+'_assessment_warning_episodes.csv'),dtype={'zip':'string'},parse_dates=['forecast_origin','matched_event_date'])
        ev=pd.read_csv(primary/(tag(model)+'_assessment_event_detection.csv'),dtype={'zip':'string'},parse_dates=['event_date'])
        alarms['qualifying_at_alarm']=[bool(daily.loc[(z,d),'qualify']) for z,d in zip(alarms.zip,alarms.forecast_origin)]
        alarms['model']=model;warningdetails.append(alarms)
        matched=alarms.loc[alarms.true_warning].copy()
        assert len(matched)==int(ev.detected.sum()) and not matched.duplicated(['zip','matched_event_date']).any()
        for h in [1,2,3]:
            a=matched.loc[matched.lead_days.eq(h)]
            leadrows.append(dict(model=model,lead_days=h,detected_events=len(a),share_of_detections=len(a)/len(matched),recall_of_all_events=len(a)/len(ev),already_qualifying=int(a.qualifying_at_alarm.sum())))
        qualified=int(matched.qualifying_at_alarm.sum());early=int((matched.lead_days>=2).sum())
        leadrows.append(dict(model=model,lead_days='SUMMARY',detected_events=len(matched),share_of_detections=1.,recall_of_all_events=len(matched)/len(ev),already_qualifying=qualified,
                             share_already_qualifying=qualified/len(matched),recall_at_least_2_days=early/len(ev),
                             recall_before_current_qualification=int((~matched.qualifying_at_alarm).sum())/len(ev)))
        ev['model']=model;eventdetails.append(ev)
    leads=pd.DataFrame(leadrows);leads.to_csv(output/'lead_time_breakdown.csv',index=False)
    warnings=pd.concat(warningdetails,ignore_index=True);warnings.to_csv(output/'primary_warning_review.csv',index=False)
    # Deterministic illustrative selection: median observed event/alert MA3 for DP within each outcome.
    # Outcomes are used for retrospective examples only; they never tune a threshold.
    dpwarnings=warnings.loc[warnings.model.eq(MODELS[0])]
    dpevents=eventdetails[0]
    pools={'true_warning':dpwarnings.loc[dpwarnings.true_warning].copy(),
           'missed_event':dpevents.loc[~dpevents.detected].copy(),
           'false_warning':dpwarnings.loc[~dpwarnings.true_warning].copy()}
    paths={'Dynamic Panel FE':root/'rolling_trajectory_outputs/rolling60_weekly_development_20261027/predictions.npz',
           'XGBoost':root/'xgboost_trajectory_outputs/xgb60_weekly_development_20261027/predictions.npz',
           'Random Forest':root/'random_forest_trajectory_outputs/rf60_weekly_development_20261027/predictions.npz'}
    arrays={m:np.load(p,allow_pickle=False) for m,p in paths.items()}
    selected=[]
    for kind,pool in pools.items():
        datecol='event_date' if kind=='missed_event' else 'forecast_origin'
        pool['selection_ma3']=[float(daily.loc[(z,d),'ma']) for z,d in zip(pool.zip,pool[datecol])]
        pool=pool.sort_values(['selection_ma3','zip',datecol]).reset_index(drop=True);row=pool.iloc[len(pool)//2]
        zipcode=row.zip;focus=pd.Timestamp(row[datecol]);origin=focus-pd.Timedelta(days=3) if kind=='missed_event' else focus
        event=focus if kind=='missed_event' else row.matched_event_date
        one=daily.loc[zipcode].loc[origin-pd.Timedelta(days=18):origin+pd.Timedelta(days=10)]
        f,ax=plt.subplots(2,1,figsize=(11,6.5),sharex=True,gridspec_kw={'height_ratios':[3,1.25]})
        ax[0].plot(one.index,one.cases_today,color='#CBD5E1',label='Recorded daily cases')
        ax[0].plot(one.index,one.ma,color='#173654',lw=2,label='Actual trailing3-day mean')
        ax[0].plot(one.index,one.threshold,color='#D94841',ls='--',label='Historical mean +2SD')
        ax[0].axhline(5/3,color='#A0AFC0',ls=':',label='Minimum3-day sum5 /3')
        ax[0].axvline(origin,color='black',ls=':',label='Illustrative forecast origin')
        if pd.notna(event):
            for axis in ax:axis.axvline(event,color='#7C3AED',ls='--',alpha=.7)
        for i,m in enumerate(MODELS):
            a=arrays[m];z=np.flatnonzero(a['zip']==zipcode)[0];j=np.flatnonzero(a['forecast_origins'].astype('datetime64[D]')==origin.to_datetime64().astype('datetime64[D]'))[0]
            pred=np.maximum(a['raw_predictions'][z,j],0)
            past=daily.loc[zipcode].loc[origin-pd.Timedelta(days=1):origin,'cases_today'].to_numpy()
            seq=np.r_[past,pred];ma3=np.convolve(seq,np.ones(3)/3,mode='valid')
            ax[0].plot(pd.date_range(origin+pd.Timedelta(days=1),periods=3),ma3,'o-',color=COLORS[i],label=m+' forecast MA3')
            ww=warnings.loc[warnings.model.eq(m)&warnings.zip.eq(zipcode)&warnings.forecast_origin.between(one.index.min(),one.index.max())]
            for _,w in ww.iterrows():ax[1].scatter(w.forecast_origin,i,marker='^' if w.true_warning else 'x',color=COLORS[i],s=65)
        ax[0].set(ylabel='Cases / mean cases',title=f'{kind.replace("_"," ").title()} for Dynamic Panel | ZIP {zipcode}')
        ax[0].legend(ncol=2,fontsize=8,loc='upper left')
        ax[1].set(yticks=range(3),yticklabels=['Dynamic Panel','XGBoost','Random Forest'],ylim=(-.6,2.6),xlabel='2020 calendar date')
        ax[1].set_title('Issued warnings: triangle=matched, cross=false; purple line=confirmation',fontsize=10)
        ax[1].xaxis.set_major_formatter(mdates.DateFormatter('%b %d'));f.autofmt_xdate();f.tight_layout();f.savefig(figures/(f'example_{kind}.png'),dpi=180);plt.close(f)
        selected.append(dict(type=kind,reference_model=MODELS[0],zip=zipcode,focus_date=str(focus.date()),illustrative_forecast_origin=str(origin.date()),event_date=str(event.date()) if pd.notna(event) else '',selection_ma3=row.selection_ma3,selection='Median reference-model outcome MA3, ties ZIP/date; retrospective illustration'))
    pd.DataFrame(selected).to_csv(output/'example_selection.csv',index=False)
    f,axes=plt.subplots(1,2,figsize=(11,4))
    x=np.arange(3);bottom=np.zeros(3)
    for h,color in zip([1,2,3],['#A0AFC0','#2563eb','#087f78']):
        v=[float(leads.loc[leads.model.eq(m)&leads.lead_days.eq(h),'recall_of_all_events'].iloc[0]) for m in MODELS]
        axes[0].bar(x,v,bottom=bottom,color=color,label=f'{h}-day lead');bottom+=v
    axes[0].set(xticks=x,xticklabels=['DP','XGB','RF'],ylabel='Recall across ALL eligible events',ylim=(0,1));axes[0].legend(fontsize=9)
    ss=leads.loc[leads.lead_days.eq('SUMMARY')].set_index('model').loc[MODELS]
    axes[1].bar(x,ss.share_already_qualifying,color=COLORS);axes[1].set(xticks=x,xticklabels=['DP','XGB','RF'],ylabel='Share of detections already qualifying at alarm',ylim=(0,1))
    f.suptitle('Confirmation lead is different from warning before current abnormal activity');f.tight_layout();f.savefig(figures/'04_lead_time_and_prior_signal.png',dpi=180);plt.close(f)
    f,axes=plt.subplots(2,2,figsize=(12,8));order=[x[0] for x in configs]
    for axis,col,title in zip(axes.ravel(),['average_precision','event_recall','warning_precision','positive_origin_share'],['Average precision','Event recall','Warning precision','Positive origin share (label prevalence)']):
        for m,color in zip(MODELS,COLORS):
            q=assess.loc[assess.model.eq(m)].set_index('scenario').loc[order];axis.plot(order,q[col],marker='o',color=color,label=m)
        axis.set(title=title);axis.grid(alpha=.2)
    axes[0,0].legend(fontsize=8);f.suptitle('One condition changes at a time; event labels differ across scenarios');f.tight_layout();f.savefig(figures/'05_definition_sensitivity.png',dpi=180);plt.close(f)
    summary=dict(primary_replicated_notebook27=True,source_hashes=source_hash,one_factor_at_a_time=[dict(scenario=n,min_cases=mc,sd_multiplier=sd) for n,mc,sd in configs],
                 primary_rule_unchanged=True,model_refit=False,later_years_used=False,
                 examples='Deterministic median severity outcomes for Dynamic Panel; all model warnings overlaid, illustrative only',
                 caution='APs across different label definitions are different tasks; do not select definition by maximum AP. No paired statistical superiority claims.',
                 before_signal='qualifying_at_alarm=False only means today fails composite gate; earlier abnormalities can exist. It does not establish warning before biological onset.',
                 threshold_policy='Each scenario uses calibration only, original nonnegative grid and <=1false alert/30at-riskZIP-days budget. Assessment never chooses thresholds.')
    (output/'review_definition.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
    report='# Notebook28：事件定义与预警稳定性\n\n## 主方案提前量\n\n'+ss.to_string()+'\n\n## 事件定义敏感性\n\n'+pd.DataFrame(catalog).to_string(index=False)+'\n\n## 9—12月评估\n\n'+assess[['scenario','model','average_precision','positive_origin_share','event_recall','warning_precision','threshold']].to_string(index=False)
    report+='\n\n主方案PRIMARY=三天至少5例、超过历史平均+2SD、连续两天确认。其它只改一个条件，未替换主定义。所有方案先4—8月校准，9—12月仅评估；未使用2021—2023年。AP随标签及正例率变化，不能跨定义挑最高AP证明方法更好。案例是按DP各结果类型MA3的中位数选取，展示三模型同一时段；它们用于解释而非证明典型性。\n\n已满足异常条件后、确认前发出的预警与在当天尚未满足条件时发出的预警需区分。qualifying_at_alarm=False不能证明历史从未出现异常，不能等同真实疫情尚未发生。提前量只对检出事件有定义，>=2天召回率以全部事件为分母。所有现有统计仍为回顾性探索，未完成独立临床/公共卫生事件验证。\n'
    (output/'RESULTS_CN.md').write_text(report,encoding='utf-8')
    print(ss.to_string());print(assess[['scenario','model','average_precision','event_recall','warning_precision']].to_string(index=False))
    return assess,leads


In [ ]:
assessment,leads=run_review(ROOT,OUTPUT)
display(assessment)
display(leads)

## Reading order
1. RESULTS_CN.md and lead_time_breakdown.csv.
2. Three example figures and example_selection.csv.
3. sensitivity_event_counts.csv and assessment_sensitivity_and_ranks.csv.
4. Scenario subfolders hold auditable event labels, warning episodes, scores, calibration grids and definitions.

Example selection uses observed outcomes for retrospective illustration, not training. Warning before confirmation is not necessarily warning before the first abnormal day. No primary threshold or model is selected from sensitivity assessment scores.

In [ ]:
import pandas as pd
from IPython.display import Image,display
display(pd.read_csv(OUTPUT/"lead_time_breakdown.csv"))
display(pd.read_csv(OUTPUT/"assessment_sensitivity_and_ranks.csv"))
for name in ["example_true_warning.png","example_missed_event.png","example_false_warning.png","04_lead_time_and_prior_signal.png","05_definition_sensitivity.png"]:
    display(Image(filename=str(OUTPUT/"figures"/name)))